In [ ]:
import pandas as pd
from datetime import datetime

from netsuite_auth import get_auth, run_suiteql, run_suiteql_all

auth, BASE_URL = get_auth()

In [ ]:
# --- Query: Sales orders with header and line detail ---
# Adjust the date range and status filter as needed.

DATE_FROM = "2025-01-01"
DATE_TO   = "2025-12-31"

rows = run_suiteql_all(f"""
    SELECT
        t.id                        AS order_id,
        t.tranid                    AS document_number,
        t.trandate,
        BUILTIN.DF(t.status)        AS status,
        BUILTIN.DF(t.entity)        AS customer,
        t.foreigntotal              AS order_total,
        tl.id                       AS line_id,
        tl.linesequencenumber       AS line_number,
        BUILTIN.DF(tl.item)         AS item_name,
        tl.quantity,
        tl.rate,
        tl.amount                   AS line_amount
    FROM transactionline tl
    INNER JOIN transaction t ON t.id = tl.transaction
    WHERE t.type = 'SalesOrd'
      AND t.trandate >= TO_DATE('{DATE_FROM}', 'YYYY-MM-DD')
      AND t.trandate <= TO_DATE('{DATE_TO}',   'YYYY-MM-DD')
      AND tl.itemtype IS NOT NULL
    ORDER BY t.trandate DESC, t.id, tl.linesequencenumber
""")

df = pd.DataFrame(rows)
if not df.empty:
    df["trandate"] = pd.to_datetime(df["trandate"])
    df[["quantity", "rate", "line_amount", "order_total"]] = (
        df[["quantity", "rate", "line_amount", "order_total"]]
        .apply(pd.to_numeric, errors="coerce")
    )

print(f"{len(df)} lines across {df['order_id'].nunique() if not df.empty else 0} orders")
df.head(20)

In [ ]:
# --- Summary: Order totals by status ---
if not df.empty:
    summary = (
        df.drop_duplicates(subset="order_id")  # one row per order for the total
        .groupby("status")
        .agg(
            order_count = ("order_id",    "count"),
            total_amount = ("order_total", "sum"),
        )
        .sort_values("total_amount", ascending=False)
        .reset_index()
    )
    summary["total_amount"] = summary["total_amount"].round(2)
    display(summary)
else:
    print("No data.")

In [ ]:
# --- Export: Save to Excel ---
if not df.empty:
    ts = datetime.now().strftime("%Y-%m-%d-%H-%M")
    output_file = f"sales_orders_{DATE_FROM}_{DATE_TO}_{ts}.xlsx"
    df.to_excel(output_file, index=False)
    print(f"Saved {len(df)} rows to {output_file}")
else:
    print("Nothing to export.")